# Tutorial 02: Data Preparation and Quality Control

**Before you start:** Run this notebook from the repo (e.g. open it under `Notebooks/tutorials/`). Replace every `path/to/...` and adjust the path template `.../(condition)/(filename).*` to match your folder layout.

**Why prepare and audit?** Microscopy batches often mix image sizes, channel counts, or misaligned masks. Quality Control (QC) catches that **before** the model runs (saving GPU time and confusing errors). The `audit_data()` method summarizes height, width, channels, intensity range, read errors, and whether each image matches its mask shape.

**Note:** You can use the E. coli toy dataset from [Tutorial 03](03_core_phenotyping_workflow.ipynb) to test these steps out-of-the-box.

See [Best Practices: Data Audit](../../docs/guides/best-practices.md#data-auditing) and [Getting Started: Image Preprocessing](../../docs/getting-started.md#image-preprocessing).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup

In [ ]:
from phenome import PhenoMe, get_metadata_from_path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

## 2. Find Files and Audit

The recommended flow: `find_files()` → `audit_data()` → interpret results → `process_images()`.

In [ ]:
# Find files (with optional mask_dir for property presets)
image_dir = "path/to/images"
mask_dir = "path/to/masks"  # Optional; omit if no masks

metadata_fn = get_metadata_from_path(".../(condition)/(filename).*")
file_df = pheno.find_files(
    image_dir,
    mask_dir=mask_dir,
    metadata_fn=metadata_fn,
    extensions=[".tif", ".tiff", ".png"],
)

print(f"Found {len(file_df)} images")

## 3. Run Audit and Interpret Results

`audit_data()` returns a DataFrame with columns: `file_path`, `height`, `width`, `channels`, `min`, `max`, `error`, `shape_match` (image-mask alignment), `mask_height`, `mask_width`, etc.

**What to look for:** Most rows should share one `(height, width, channels)`. Many different shapes → fix acquisition or use `pad_size` / consistent cropping. Non-empty `error` → broken paths or unreadable files. `shape_match` is `False` → image and mask grids do not match (fix filenames or alignment before mask-based features).

| Finding | Typical action |
|---------|----------------|
| One dominant shape | Default `resize_size=224` is often fine |
| Very small images | Consider `pad_size` before resize |
| Multi-channel fluorescence | `channel_mode='split'` |
| RGB / brightfield | `channel_mode='combined'` |

Use the audit to choose:
- **resize_size**: Target size for model input (default 224)
- **pad_size**: Pad small images before resizing
- **channel_mode**: `'split'` for fluorescence, `'combined'` for brightfield/RGB

In [ ]:
audit_df = pheno.audit_data()
print(audit_df.head(10))

## 5. Proceed to Processing

After auditing, run `process_images()` with appropriate parameters. See [Common Workflows](../../docs/examples/workflows.md) for full examples.

In [ ]:
# Example: process with parameters informed by audit
# from phenome import load_dinov2_model
# model, wrapper = load_dinov2_model(device=device)
# pheno.process_images(wrapper, resize_size=224, channel_mode="split")

## Next Steps

- [03_core_phenotyping_workflow.ipynb](03_core_phenotyping_workflow.ipynb): Core API tutorial
- [04_advanced_metadata_handling.ipynb](04_advanced_metadata_handling.ipynb): Handling messy data
- [05_exploratory_analysis_and_explainability.ipynb](05_exploratory_analysis_and_explainability.ipynb): Advanced analysis after successful QC and embedding
- [Best Practices](../../docs/guides/best-practices.md): Reproducibility and error handling